In [3]:
import pandas as pd
import numpy as np
import json
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve
from sklearn.preprocessing import StandardScaler



df = pd.read_csv('rfm_modeling_snapshot.csv')

print(f"Max snapshot date in data: {df['snapshot_date'].max()}")

drop_cols = ['customer_id', 'snapshot_date', 'split', 'churn_next_60d']

df_encoded = pd.get_dummies(df, columns=['city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier', 'preferred_category', 'marketing_consent'], drop_first=True)

X = df_encoded.drop(columns=[c for c in drop_cols if c in df_encoded.columns])
y = df_encoded['churn_next_60d']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("--- Baseline Model: Logistic Regression ---")
baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train_scaled, y_train)
base_preds = baseline.predict(X_test_scaled)
print(classification_report(y_test, base_preds))

print("--- Champion Model: XGBoost ---")
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
xgb.fit(X_train, y_train)
xgb_probs = xgb.predict_proba(X_test)[:, 1]

CUSTOM_THRESHOLD = 0.35
xgb_preds = (xgb_probs >= CUSTOM_THRESHOLD).astype(int)
print(f"XGBoost Metrics at Threshold {CUSTOM_THRESHOLD}:")
print(classification_report(y_test, xgb_preds))

metrics = {
    "model": "XGBoost",
    "threshold_used": CUSTOM_THRESHOLD,
    "accuracy": float(np.mean(xgb_preds == y_test)),
    "confusion_matrix": confusion_matrix(y_test, xgb_preds).tolist()
}

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

model_artifact = {
    "model": xgb,
    "features": list(X.columns)
}
with open("model.pkl", "wb") as f:
    pickle.dump(model_artifact, f)

print("\nSuccess! Saved model.pkl and metrics.json")

Max snapshot date in data: 2025-09-30
--- Baseline Model: Logistic Regression ---
              precision    recall  f1-score   support

           0       0.80      0.83      0.81       255
           1       0.80      0.76      0.78       225

    accuracy                           0.80       480
   macro avg       0.80      0.80      0.80       480
weighted avg       0.80      0.80      0.80       480

--- Champion Model: XGBoost ---
XGBoost Metrics at Threshold 0.35:
              precision    recall  f1-score   support

           0       0.82      0.72      0.77       255
           1       0.72      0.82      0.77       225

    accuracy                           0.77       480
   macro avg       0.77      0.77      0.77       480
weighted avg       0.77      0.77      0.77       480


Success! Saved model.pkl and metrics.json


In [ ]:
from google.colab import drive
drive.mount('/content/drive')